<a href="https://colab.research.google.com/github/Amulyanrao7777/NLP/blob/main/lab7_assignmentcopy_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###[Question](https://docs.google.com/document/d/1de_6x57DD4sU3aFcq_3pdPYfHhdI4G60XPSTG9b_D9o/edit?usp=sharing)

In [35]:
!pip install gensim scikit-learn numpy

In [36]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [37]:
import re
from gensim.models import Word2Vec

# 1. Prepare a toy dataset
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the dog barked at the mailman",
    "the cat sat on the mat",
    "dogs and cats are popular pets"
]

# 2. Tokenization: clean and split sentences into words
# from nltk.corpus import stopwords
# stop_words = set(stopwords.words('english'))
# stop_words -= {"not", "no", "never", "still", "but", "non"}

# def tokenize(text):
#     text = text.lower()
#     text = re.sub(r'[^a-z\s]', '', text)
#     return [w for w in text.split() if w not in stop_words]

# tokenized_corpus = [tokenize(sentence) for sentence in corpus]
# print("Tokenized corpus:")
# for tokens in tokenized_corpus:
#     print(" ", tokens)
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

tokenized_corpus = [tokenize(sentence) for sentence in corpus]
print("Tokenized corpus:")
for tokens in tokenized_corpus:
    print(" ", tokens)

Tokenized corpus:
  ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
  ['the', 'dog', 'barked', 'at', 'the', 'mailman']
  ['the', 'cat', 'sat', 'on', 'the', 'mat']
  ['dogs', 'and', 'cats', 'are', 'popular', 'pets']


In [38]:
# 3. Train the Word2Vec model
#   vector_size = dimensionality of the word vectors
#   window      = max distance between current and predicted word
#   min_count   = ignore words with frequency below this
#   sg          = 1 for Skip-gram, 0 for CBOW

print("Training Word2Vec model...")
w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=10,
    window=2,
    min_count=1,
    sg=1  # Skip-gram
)

# 4. Explore the embeddings
word = "dog"
print(f"\nVector representation of '{word}':\n", w2v_model.wv[word])

# Find most similar words
print(f"\nWords most similar to '{word}':")
similar_words = w2v_model.wv.most_similar(word, topn=2)
for w, score in similar_words:
    print(f"  - {w} (score: {score:.4f})")

Training Word2Vec model...

Vector representation of 'dog':
 [ 0.07381444 -0.01533606 -0.04537103  0.06552491 -0.04859608 -0.01815614
  0.02876893  0.0099145  -0.0828473  -0.0944812 ]

Words most similar to 'dog':
  - the (score: 0.5436)
  - cat (score: 0.3587)


In [39]:
import gensim.downloader as api

# 1. Load pre-trained GloVe (50-dimensional, trained on Wikipedia + Gigaword)
# First run downloads ~66MB
print("Loading pre-trained GloVe model (this may take a moment)...")
glove_model = api.load("glove-wiki-gigaword-50")
print("Done!")

Loading pre-trained GloVe model (this may take a moment)...
Done!


In [40]:
# 2. The famous vector arithmetic: King - Man + Woman = ?
print("Solving the analogy: King is to Man as [?] is to Woman")
result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(f"Answer: {result[0][0]}  (Confidence: {result[0][1]:.4f})")

# 3. Word similarity
sim = glove_model.similarity('apples', 'oranges')
print(f"\nSimilarity between 'apples' and 'oranges': {sim:.4f}")

Solving the analogy: King is to Man as [?] is to Woman
Answer: queen  (Confidence: 0.8524)

Similarity between 'apples' and 'oranges': 0.8594


In [41]:
from gensim.models import FastText

# Train FastText on the same toy corpus
print("Training FastText model...")
ft_model = FastText(
    sentences=tokenized_corpus,
    vector_size=10,
    window=2,
    min_count=1
)
print("Done!")

Training FastText model...
Done!


In [42]:
# Test on an Out-Of-Vocabulary (OOV) word!
# "doggy" is NOT in our training corpus — Word2Vec would crash here.
oov_word = "doggy"

try:
    vector = ft_model.wv[oov_word]
    print(f"Successfully generated vector for unknown word '{oov_word}':")
    print(vector)

    sims = ft_model.wv.most_similar(oov_word, topn=2)
    print(f"\nWords similar to '{oov_word}':", sims)
except KeyError:
    print(f"Error: '{oov_word}' not in vocabulary.")

Successfully generated vector for unknown word 'doggy':
[ 0.0040328   0.00336035 -0.00267577 -0.01673043 -0.0090626   0.00541232
  0.02354874  0.00272812  0.00677915  0.01575618]

Words similar to 'doggy': [('and', 0.7510858178138733), ('barked', 0.5487850904464722)]


---
## Assignment: Compare Embedding Models on Tech Support Data

**Objective:** Build and compare two **Sentiment Classifiers** (Resolved vs. Frustrated) using different word embeddings.

**Pipeline:** Text -> Tokens -> Embeddings -> Classifier

### Step 1: Build the Dataset

In [43]:
# 10 resolved (label=1) and 10 frustrated (label=0) tech support statements
sentences = [
    # Resolved (1)
    "The new patch fixed my router thanks",
    "Customer service was incredibly helpful",
    "My issue was resolved within minutes amazing support",
    "The technician walked me through everything perfectly",
    "Update worked flawlessly my device runs great now",
    "Thank you the reset instructions were very clear",
    "Problem solved the wifi is back to full speed",
    "Support team was friendly and fixed it quickly",
    "Everything works now really appreciate the fast response",
    "Great service my account is fully restored",
    # Frustrated (0)
    "My software keeps crashing after the update",
    "I have been on hold for 45 minutes",
    "This is completely unacceptable my internet is still down",
    "Nobody is helping me and my issue has not been fixed",
    "The app crashes every single time I open it",
    "I have contacted support three times and nothing works",
    "Terrible service I want a refund now",
    "Why is the update breaking everything this is ridiculous",
    "Still waiting for a callback that was promised yesterday",
    "My data was lost after the migration and no one cares",
]

labels = [1]*10 + [0]*10

print(f"Dataset size: {len(sentences)} sentences")
print(f"Labels: {labels}")

Dataset size: 20 sentences
Labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### Step 2: Load Embedding Models

In [44]:
import gensim.downloader as api

# Model A: GloVe trained on Twitter data (25-dim)
# Twitter data is well-suited for conversational/support text
print("Loading glove-twitter-25 (first run downloads ~48MB)...")
glove_twitter = api.load("glove-twitter-25")
print("GloVe Twitter loaded!")

Loading glove-twitter-25 (first run downloads ~48MB)...
GloVe Twitter loaded!


In [45]:
# Model B: FastText trained on our small dataset
from gensim.models import FastText

tokenized_support = [tokenize(s) for s in sentences]

print("Training FastText on support dataset...")
ft_support = FastText(
    sentences=tokenized_support,
    vector_size=25,   # Match GloVe dims for fair comparison
    window=3,
    min_count=1,
    epochs=50
)
print("FastText trained!")

Training FastText on support dataset...
FastText trained!


### Step 3: Sentence Vectorization (Average of Word Vectors)

In [46]:
import numpy as np

def sentence_vector(sentence, model, vector_size):
    """
    Convert a sentence to a fixed-size vector by averaging its word vectors.
    OOV words are skipped. If no words have vectors, return a zero vector.
    """
    tokens = tokenize(sentence)
    vectors = []

    for token in tokens:
        try:
            vectors.append(model[token])
        except KeyError:
            pass  # Skip OOV words (GloVe doesn't handle them natively)

    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)  # Fallback for all-OOV sentences


# Build feature matrices
GLOVE_DIM = 25
FT_DIM    = 25

X_glove = np.array([sentence_vector(s, glove_twitter,    GLOVE_DIM) for s in sentences])
X_ft    = np.array([sentence_vector(s, ft_support.wv,    FT_DIM)    for s in sentences])

print("GloVe feature matrix shape:",   X_glove.shape)
print("FastText feature matrix shape:", X_ft.shape)

GloVe feature matrix shape: (20, 25)
FastText feature matrix shape: (20, 25)


### Step 4: Train Logistic Regression Classifiers

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Train on full dataset (small dataset — CV gives a better estimate than a single split)
clf_glove = LogisticRegression(max_iter=500, random_state=42)
clf_ft    = LogisticRegression(max_iter=500, random_state=42)

clf_glove.fit(X_glove, labels)
clf_ft.fit(X_ft,    labels)

# Cross-validation accuracy
cv_glove = cross_val_score(clf_glove, X_glove, labels, cv=5).mean()
cv_ft    = cross_val_score(clf_ft,    X_ft,    labels, cv=5).mean()

print(f"GloVe Twitter-25  | 5-fold CV Accuracy: {cv_glove:.2%}")
print(f"FastText (custom) | 5-fold CV Accuracy: {cv_ft:.2%}")

GloVe Twitter-25  | 5-fold CV Accuracy: 85.00%
FastText (custom) | 5-fold CV Accuracy: 70.00%


### Step 5: Test on a Tricky New Tweet

In [48]:
test_tweets = [
    "The wifi is behaving weirdly today, what is going on?",
    "Finally sorted! The agent was super patient with me.",
    "Three days and still no fix, this is beyond frustrating.",
]

label_map = {1: "Resolved", 0: "Frustrated"}

print(f"{'Tweet':<55} {'GloVe':^15} {'FastText':^15}")
print("-" * 85)

for tweet in test_tweets:
    v_glove = sentence_vector(tweet, glove_twitter, GLOVE_DIM).reshape(1, -1)
    v_ft    = sentence_vector(tweet, ft_support.wv, FT_DIM).reshape(1, -1)

    pred_glove = label_map[clf_glove.predict(v_glove)[0]]
    pred_ft    = label_map[clf_ft.predict(v_ft)[0]]

    print(f"{tweet[:54]:<55} {pred_glove:^15} {pred_ft:^15}")

Tweet                                                        GloVe         FastText    
-------------------------------------------------------------------------------------
The wifi is behaving weirdly today, what is going on?     Frustrated      Frustrated   
Finally sorted! The agent was super patient with me.      Frustrated       Resolved    
Three days and still no fix, this is beyond frustratin    Frustrated      Frustrated   


---
##Discussion

| Embedding | Training Data | Strength in This Task |
|---|---|---|
| **GloVe Twitter-25** | 2B tweets | Captures informal language, slang, abbreviations common in support chats |
| **FastText (custom)** | 20 sentences | Handles typos and word variants via subwords, but limited vocabulary on tiny corpus |

**Key Takeaway:** Even with perfect embeddings, accuracy on a 20-sentence dataset is inherently noisy. The goal is to understand the **pipeline**: Text -> Tokens -> Embeddings -> Classifier — and to see how different embedding *sources* (Twitter vs. custom) affect real-world text.